<a href="https://colab.research.google.com/github/jempio/cog403-final-project/blob/main/COG403_Project_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Appraisal-Text Fusion for Emotion Classification

## 1) Research Question and Hypotheses

### Research Question
Does combining text and appraisal information to predict emotion outperform using either source alone?

### Hypotheses

**H1. Appraisal-only information is a predictor of emotion.**  
The seven appraisal dimensions (Attention, Certainty, Effort, Pleasant, Responsibility, Control, Circumstance) are theoretically tied to emotional experience, so a simple classifier trained only on these values is expected to perform above chance.

**H2. Text-only features should also perform well.**  
A TF-IDF representation of the sentence paired with logistic regression should provide a strong baseline for emotion classification.

**H3. Fusion should outperform single-source models.**  
Combining text features with appraisal features should produce better **macro F1** than either text-only or appraisal-only models, because the model can use both semantic content and structured affective cues.

## 2) Methods

### 2.1) Dataset
We use the Appraisal enISEAR dataset, where each instance contains:

- a sentence describing an emotional event
- an emotion class
- seven appraisal annotations

**Text input**  
The `Sentence` column is encoded with TF-IDF for a classical baseline

**Appraisal input**  
The seven appraisal columns are treated as a compact numeric feature vector:

- `Attention`
- `Certainty`
- `Effort`
- `Pleasant`
- `Responsibility`
- `Control`
- `Circumstance`

**Fusion**  
The text representation and appraisal vector are concatenated and passed to a classifier.

**Output**  
The target is the emotion label in `Prior_Emotion` (used here as the dataset's emotion-class column).

**Training objective**  
Single-label multiclass classification using cross-entropy loss.

### 2.2) Evaluation
We report:

- **Macro F1**, our main metric(important because classes may not be perfectly balanced)
- **Accuracy**

## 3) Data Cleaning

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

In [16]:
TEXT_COL = "Sentence"
LABEL_COL = "Prior_Emotion"
APPRAISAL_COLS = [
    "Attention",
    "Certainty",
    "Effort",
    "Pleasant",
    "Responsibility",
    "Control",
    "Circumstance",
]

RANDOM_STATE = 42
TEST_SIZE = 0.20
results = []

In [17]:
data_path = "emotion_appraisal_corpus.tsv"

raw_df = pd.read_csv(data_path, sep="\t")
print(f"Loaded dataset from: {data_path}")
print(f"Shape: {raw_df.shape}")
display(raw_df.head())

Loaded dataset from: emotion_appraisal_corpus.tsv
Shape: (1001, 10)


,Sentence_id,Prior_Emotion,Sentence,Attention,Certainty,Effort,Pleasant,Responsibility,Control,Circumstance
0,271,Fear,"I felt ... when my 2 year old broke her leg, a...",3,1,3,0,0,1,1
1,597,Shame,I felt ... one Christmas as one of our patient...,3,3,1,0,2,0,0
2,282,Guilt,I felt ... because I could not help a friend w...,1,3,0,0,3,2,1
3,171,Disgust,I felt ... when I read that hunters had killed...,2,3,0,0,0,0,0
4,509,Sadness,I felt ... when my Gran passed away.,2,2,2,0,0,0,3


In [18]:
# basic cleaning

df = raw_df.copy()

df = df.dropna(subset=[TEXT_COL, LABEL_COL])
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

for col in APPRAISAL_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=APPRAISAL_COLS)
df = df.reset_index(drop=True)

print(df.shape)
display(df.head())

(1001, 10)


,Sentence_id,Prior_Emotion,Sentence,Attention,Certainty,Effort,Pleasant,Responsibility,Control,Circumstance
0,271,Fear,"I felt ... when my 2 year old broke her leg, a...",3,1,3,0,0,1,1
1,597,Shame,I felt ... one Christmas as one of our patient...,3,3,1,0,2,0,0
2,282,Guilt,I felt ... because I could not help a friend w...,1,3,0,0,3,2,1
3,171,Disgust,I felt ... when I read that hunters had killed...,2,3,0,0,0,0,0
4,509,Sadness,I felt ... when my Gran passed away.,2,2,2,0,0,0,3


### 3.1) Exploratory Data Analysis

In [19]:
# quick eda

print("Dataset shape:", df.shape)

print("\nClass counts:")
print(df[LABEL_COL].value_counts())

df["text_len_words"] = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))
print("\nText length summary (words):")
display(df["text_len_words"].describe())

Dataset shape: (1001, 10)

Class counts:
Prior_Emotion
Fear       143
Shame      143
Guilt      143
Disgust    143
Sadness    143
Anger      143
Joy        143
Name: count, dtype: int64

Text length summary (words):


,text_len_words
count,1001.000000
mean,25.301698
std,19.900525
min,6.000000
25%,14.000000
50%,20.000000
75%,30.000000
max,270.000000


### 3.1) Test/Train Split

We use a stratified split so that the emotion-class distribution is preserved in both the training and test sets.  
We also encode the string labels into integer IDs for models that require numeric targets.

In [20]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE,random_state=RANDOM_STATE, stratify=df[LABEL_COL])

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df[LABEL_COL])
y_test = label_encoder.transform(test_df[LABEL_COL])

X_train_text = train_df[TEXT_COL].tolist()
X_test_text = test_df[TEXT_COL].tolist()

X_train_app = train_df[APPRAISAL_COLS].to_numpy(dtype=float)
X_test_app = test_df[APPRAISAL_COLS].to_numpy(dtype=float)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Classes:", list(label_encoder.classes_))

Train shape: (800, 11)
Test shape: (201, 11)
Classes: ['Anger', 'Disgust', 'Fear', 'Guilt', 'Joy', 'Sadness', 'Shame']


## 4) Models

In [21]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

### 4.1) Dummy model

In [22]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_text, y_train)
dummy_predictions = dummy.predict(X_test_text)

### 4.2) Baseline 1: Appraisal-only Sparse Logistic Regression

In [23]:
app_scaler = StandardScaler()
X_train_app_scaled = app_scaler.fit_transform(X_train_app)
X_test_app_scaled = app_scaler.fit_transform(X_test_app)

appraisal = LogisticRegression(penalty='l1', solver='liblinear', C=0.1)
appraisal.fit(X_train_app_scaled, y_train)
appraisal_predictions = appraisal.predict(X_test_app_scaled)

### 4.3) Baseline 2: TF-IDF Sparse Logistic Regression

In [24]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

text = LogisticRegression(penalty='l1', solver='saga', C=0.1)
text.fit(X_train_tfidf, y_train)
text_predictions = text.predict(X_test_tfidf)

### 4.4) Main Model

In [25]:
X_train_app_sparse = csr_matrix(X_train_app_scaled)
X_test_app_sparse = csr_matrix(X_test_app_scaled)

X_train_fused = hstack([X_train_tfidf, X_train_app_sparse], format="csr")
X_test_fused = hstack([X_test_tfidf, X_test_app_sparse], format="csr")

fused = LogisticRegression(penalty='l1', solver='saga', C=0.1)

fused.fit(X_train_fused, y_train)
fused_prediction = fused.predict(X_test_fused)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
